# IFRS 17 roll-forward
Run every cell after installing with the ui extra and dev group. The public invented Term book moves from 31 December 2025 to 30 June 2026. This is a GMM illustration of BEL, RA, CSM and loss components, not a full insurance revenue/expense statement.


In [ ]:
from pathlib import Path
from tempfile import mkdtemp

import plotly.graph_objects as go
import polars as pl

from avk.api import check, run
from avk.demo.generate import generate

folder = Path(mkdtemp(prefix="avk-notebook-"))
generate(folder, "TERM", 10000, 209)
print(f"Invented inputs and results: {folder}")


def execute(mode):
    config = folder / f"{mode}.json"
    check(config)
    return run(config)


def aggregate(result):
    path = next(p for p in result.files if p.name.endswith("_aggregate_output.csv"))
    return pl.read_csv(path)

The generated roll-forward file supplies opening/closing policy inventories, locked-in cohort curves and opening group balances. Profitable and onerous groups remain separate. CSM release is stored as a positive amount and subtracted in the bridge.


In [ ]:
result = execute("rollforward")
bridge = pl.read_csv(next(p for p in result.files if p.name == "IFRS17_rollforward.csv"))
assert bridge.height > 0
columns = [
    "csm_opening",
    "csm_new_business",
    "csm_interest",
    "csm_experience",
    "csm_assumptions",
    "csm_release",
    "csm_closing",
]
totals = {c: bridge[c].sum() for c in columns}
print(pl.DataFrame([totals]))
fig = go.Figure(
    go.Waterfall(
        x=columns,
        measure=["absolute", "relative", "relative", "relative", "relative", "relative", "total"],
        y=[totals[c] * (-1 if c == "csm_release" else 1) for c in columns],
    )
)
fig.update_layout(title="Six-month CSM bridge", yaxis_title="Currency units")
fig.show()

Inspect loss components and held reinsurance separately; do not net the CSM and loss component into a single measure. Coverage-unit releases describe service. Experience and assumption effects are the supported bridge attribution, not a general analysis-of-change engine.


In [ ]:
print(
    bridge.select(
        "portfolio",
        "cohort",
        "profitability",
        "loss_component_closing",
        "loss_recovery_component_closing",
        "bel_current",
        "ra_current",
    )
)
fig = go.Figure()
for column in ("csm_closing", "loss_component_closing", "ra_current"):
    fig.add_bar(x=[str(i) for i in range(bridge.height)], y=bridge[column].to_list(), name=column)
fig.update_layout(
    title="Closing group measures",
    xaxis_title="Group row",
    yaxis_title="Currency units",
    barmode="group",
)
fig.show()